# 400 YouDo 2: Matrix Factorization with Gradient Descent

This notebook solves the sparse rating matrix problem from `400_YouDo_2.ipynb`.

Goal:

- Read `ratings_long.csv`
- Build the sparse rating matrix `R` with shape `20 x 1000`
- Approximate it as `R ~= U @ V`
- Learn `U` and `V` with gradient descent
- Add L2 regularization to reduce overfitting

Important note: the squared loss is convex in `U` when `V` is fixed, and convex in `V` when `U` is fixed. Optimizing both together is a matrix-factorization problem, so gradient descent searches for a good local solution.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

ratings = pd.read_csv("ratings_long.csv")
ratings.head()

## 1. Build the Sparse Rating Matrix

The data is stored in long format: one row per known `(user, movie, rating)` triple.

The full matrix has 20 users and 1000 movies, but only a small number of entries are known. Missing ratings are stored as `NaN` and are not used in the loss.

In [ ]:
n_users = 20
n_movies = 1000
latent_dim = 4

R = np.full((n_users, n_movies), fill_value=np.nan)

for row in ratings.itertuples(index=False):
    R[row.userId, row.movieId] = row.rating

observed_mask = ~np.isnan(R)
n_observed = observed_mask.sum()
sparsity = 1 - n_observed / R.size

print("R shape:", R.shape)
print("Known ratings:", n_observed)
print("Sparsity:", round(sparsity, 4))

pd.DataFrame(R).iloc[:5, :20]

## 2. Loss Function

For known ratings only:

$$
\hat{R}_{ij} = U_i V_j
$$

$$
L(U,V) = \frac{1}{N} \sum_{(i,j) \in \Omega} (U_i V_j - R_{ij})^2 + \frac{\lambda}{N}(||U||_2^2 + ||V||_2^2)
$$

Where:

- `Omega` is the set of observed ratings
- `N` is the number of observed ratings
- `lambda` controls L2 regularization
- `U` has shape `20 x 4`
- `V` has shape `4 x 1000`

In [ ]:
user_ids = ratings["userId"].to_numpy()
movie_ids = ratings["movieId"].to_numpy()
y_true = ratings["rating"].to_numpy(dtype=float)


def predict_known(U, V, users=user_ids, movies=movie_ids):
    return (U[users] * V[:, movies].T).sum(axis=1)


def mse_loss(U, V, lambda_reg=0.1):
    y_pred = predict_known(U, V)
    error = y_pred - y_true
    mse = np.mean(error ** 2)
    penalty = lambda_reg / len(y_true) * (np.sum(U ** 2) + np.sum(V ** 2))
    return mse + penalty


def rmse_on_known(U, V):
    y_pred = predict_known(U, V)
    return np.sqrt(np.mean((y_pred - y_true) ** 2))

## 3. Gradient Descent

We initialize `U` and `V` randomly. At each epoch:

1. Predict known ratings
2. Compute prediction error
3. Compute gradients for `U` and `V`
4. Add L2 regularization gradients
5. Update parameters with gradient descent

The update rule is:

$$
U = U - \eta \nabla_U L
$$

$$
V = V - \eta \nabla_V L
$$

In [ ]:
def fit_matrix_factorization(
    users,
    movies,
    ratings_values,
    n_users=20,
    n_movies=1000,
    latent_dim=4,
    learning_rate=0.4,
    lambda_reg=0.1,
    n_epochs=5000,
    random_state=42,
):
    rng = np.random.default_rng(random_state)
    U = 0.5 * rng.normal(size=(n_users, latent_dim))
    V = 0.5 * rng.normal(size=(latent_dim, n_movies))

    history = []
    n = len(ratings_values)

    for epoch in range(n_epochs):
        predictions = (U[users] * V[:, movies].T).sum(axis=1)
        errors = predictions - ratings_values

        grad_U = np.zeros_like(U)
        grad_V = np.zeros_like(V)

        scaled_errors = 2 * errors / n
        np.add.at(grad_U, users, scaled_errors[:, None] * V[:, movies].T)
        np.add.at(grad_V.T, movies, scaled_errors[:, None] * U[users])

        grad_U += 2 * lambda_reg / n * U
        grad_V += 2 * lambda_reg / n * V

        U -= learning_rate * grad_U
        V -= learning_rate * grad_V

        if epoch % 100 == 0 or epoch == n_epochs - 1:
            mse = np.mean(errors ** 2)
            penalty = lambda_reg / n * (np.sum(U ** 2) + np.sum(V ** 2))
            history.append({
                "epoch": epoch,
                "loss": mse + penalty,
                "rmse": np.sqrt(mse),
            })

    return U, V, pd.DataFrame(history)


U, V, history = fit_matrix_factorization(
    user_ids,
    movie_ids,
    y_true,
    n_users=n_users,
    n_movies=n_movies,
    latent_dim=latent_dim,
    learning_rate=0.4,
    lambda_reg=0.1,
    n_epochs=5000,
    random_state=RANDOM_STATE,
)

history.tail()

## 4. Training Curve

If gradient descent works, the loss should go down over epochs.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["epoch"], history["loss"], color="#4c78a8")
axes[0].set_title("Regularized loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")

axes[1].plot(history["epoch"], history["rmse"], color="#f58518")
axes[1].set_title("RMSE on known ratings")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("RMSE")

plt.tight_layout()
plt.show()

print("Final regularized loss:", round(history.iloc[-1]["loss"], 4))
print("Final RMSE on known ratings:", round(history.iloc[-1]["rmse"], 4))

## 5. Compare Actual and Predicted Ratings

The learned matrices can now estimate known and unknown ratings. Below we compare predictions with the observed ratings.

In [ ]:
known_predictions = predict_known(U, V)
comparison = ratings.copy()
comparison["predicted_rating"] = known_predictions
comparison["predicted_rating_clipped"] = np.clip(known_predictions, 1, 5)
comparison["absolute_error"] = (comparison["predicted_rating"] - comparison["rating"]).abs()

comparison.head(15)

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(comparison["rating"], comparison["predicted_rating_clipped"], alpha=0.65, color="#54a24b")
plt.plot([1, 5], [1, 5], linestyle="--", color="#e45756")
plt.xlabel("Actual rating")
plt.ylabel("Predicted rating, clipped to 1-5")
plt.title("Actual vs predicted ratings")
plt.show()

## 6. Fill the Matrix and Recommend Movies

`U @ V` produces a dense `20 x 1000` matrix. That means we can estimate ratings for movies a user has not rated yet.

Below, we recommend the highest predicted unrated movies for one user.

In [ ]:
R_hat = U @ V
R_hat_clipped = np.clip(R_hat, 1, 5)

user_id = 0
already_rated = observed_mask[user_id]
unrated_movie_ids = np.where(~already_rated)[0]

recommendations = pd.DataFrame({
    "movieId": unrated_movie_ids,
    "predicted_rating": R_hat_clipped[user_id, unrated_movie_ids],
}).sort_values("predicted_rating", ascending=False)

recommendations.head(10)

## Summary

- The sparse rating matrix was represented as `R ~= U @ V`.
- `U` has shape `20 x 4` and stores user latent factors.
- `V` has shape `4 x 1000` and stores movie latent factors.
- The model was trained with gradient descent.
- L2 regularization was added to discourage overly large parameter values.
- The final dense matrix can be used to predict missing ratings and generate recommendations.